# M2.2 — A problem found in development: the raw-feature model barely converges

**The symptom, from the J1 notebook** ([01_supervised_fraud.ipynb](01_supervised_fraud.ipynb), §5.3 output):

```
XGBoost (engineered): 117 trees after early stopping, fitted in 19s, threshold 0.506
XGBoost (raw):        1886 trees after early stopping, fitted in 258s, threshold 0.918
```

The raw-feature model (the six untransformed columns, used only for the ablation in J1 §6.2) was fitted with the
hyper-parameters tuned for the *engineered* features (J1 notebook, §5.3 says so openly). It stopped at 1,886 trees when
the ceiling was 2,000: early stopping fired only 114 trees before the cap, so the cap, not the data, may have ended
training. It is also 13 times slower than the engineered model.

**Why it matters.** The raw-feature model is the "does the model work without the simulator's bookkeeping features?"
check. A model that may be under-trained makes that check unreliable, and the ablation table in the README quotes its
PR-AUC (0.9289).

**Plan, written before running anything.**
1. Reproduce the symptom and look at the validation curve: is it still improving at the end?
2. Define *converged* as: early stopping fires at no more than **half the ceiling (≤ 1,000 trees)**.
3. Try four configurations with a larger learning rate and depth (the raw features need interactions that shallow trees
   at a small learning rate build slowly): depth {4, 8} × learning rate {0.1, 0.3}. Choose, **on validation**, the
   configuration with the highest PR-AUC among those that converge.
4. Evaluate before and after on the test set once each, and compare with the committed ablation number.

## 1. Setup and data

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score

ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))

from fraud.data import SEED, load_paysim, scope_transfer_cashout, split_70_15_15
from fraud.features import make_features
from fraud.metrics import best_f1_threshold, evaluate
from fraud.model import MAX_TREES, baseline_params, fit_xgb

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

d = scope_transfer_cashout(load_paysim())
X, RAW, ENG = make_features(d)
y = d["isFraud"].to_numpy()
train_idx, val_idx, test_idx = split_70_15_15(y)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(f"raw features ({len(RAW)}): {RAW}")


def run(params, max_trees=MAX_TREES):
    """Fit the raw-feature model; return it with its validation scores and max-F1 threshold."""
    model, secs = fit_xgb(X.iloc[train_idx][RAW], y_train, X.iloc[val_idx][RAW], y_val, params, max_trees=max_trees)
    val_score = model.predict_proba(X.iloc[val_idx][RAW])[:, 1]
    return {"model": model, "secs": secs, "trees": model.best_iteration + 1, "val_score": val_score,
            "thr": best_f1_threshold(y_val, val_score)}


def test_row(label, r):
    """Test-set metrics for a fitted raw-feature run, at its validation-chosen threshold."""
    score = r["model"].predict_proba(X.iloc[test_idx][RAW])[:, 1]
    return evaluate(label, y_test, score, r["thr"], "raw", r["secs"])

raw features (6): ['isTransfer', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']


## 2. Reproduce the symptom
Same hyper-parameters as the J1 raw-feature model (the rank-1 configuration of the J1 search), ceiling of 2,000 trees.

In [2]:
PARAMS = baseline_params()
before = run(PARAMS)
curve = np.array(before["model"].evals_result()["validation_0"]["aucpr"])
print(f"{before['trees']} trees (ceiling {MAX_TREES}), fitted in {before['secs']:.0f}s, threshold {before['thr']:.3f}, "
      f"validation PR-AUC {average_precision_score(y_val, before['val_score']):.4f}")

rounds = [100, 250, 500, 1000, 1500, before["trees"]]
display(pd.DataFrame({"boosting round": rounds, "validation PR-AUC (XGBoost's aucpr)": [curve[r - 1] for r in rounds]})
        .style.format({"boosting round": "{:,}", "validation PR-AUC (XGBoost's aucpr)": "{:.4f}"}).hide(axis="index"))

1886 trees (ceiling 2000), fitted in 536s, threshold 0.918, validation PR-AUC 0.9298


boosting round,validation PR-AUC (XGBoost's aucpr)
100,0.8038
250,0.8469
500,0.8907
"1,000",0.9202
"1,500",0.9273
"1,886",0.9298


In [3]:
last = curve[-300:]
display(Markdown(f"Over the last 300 rounds the validation curve moved from {last[0]:.4f} to {last[-1]:.4f} "
                 f"(a change of {last[-1] - last[0]:+.4f}); over the first 500 rounds it moved from {curve[0]:.4f} to {curve[499]:.4f}."))

Over the last 300 rounds the validation curve moved from 0.9286 to 0.9296 (a change of +0.0011); over the first 500 rounds it moved from 0.0680 to 0.8907.

## 3. Candidate fixes

In [4]:
rows, runs = [], {}
for depth in (4, 8):
    for lr in (0.1, 0.3):
        params = {**PARAMS, "max_depth": depth, "learning_rate": lr}
        r = run(params)
        runs[(depth, lr)] = (params, r)
        rows.append({"max_depth": depth, "learning_rate": lr, "trees": r["trees"], "fit seconds": r["secs"],
                     "val PR-AUC": average_precision_score(y_val, r["val_score"]), "converged (<= half the ceiling)": r["trees"] <= MAX_TREES // 2})
rows.append({"max_depth": PARAMS["max_depth"], "learning_rate": PARAMS["learning_rate"], "trees": before["trees"],
             "fit seconds": before["secs"], "val PR-AUC": average_precision_score(y_val, before["val_score"]),
             "converged (<= half the ceiling)": before["trees"] <= MAX_TREES // 2})
cand = pd.DataFrame(rows)
display(cand.style.format({"trees": "{:,}", "fit seconds": "{:.0f}", "val PR-AUC": "{:.4f}"}).hide(axis="index"))

max_depth,learning_rate,trees,fit seconds,val PR-AUC,converged (<= half the ceiling)
4,0.100000,733,238,0.9319,True
4,0.300000,326,106,0.9285,True
8,0.100000,302,107,0.9263,True
8,0.300000,210,84,0.9270,True
4,0.030000,"1,886",536,0.9298,False


In [5]:
ok = cand[cand["converged (<= half the ceiling)"]].sort_values("val PR-AUC", ascending=False)
assert len(ok), "no candidate converged; widen the grid"
win = ok.iloc[0]
win_params, after = runs[(int(win["max_depth"]), float(win["learning_rate"]))]
display(Markdown(f"Chosen on validation: depth {win_params['max_depth']}, learning rate {win_params['learning_rate']} "
                 f"({after['trees']} trees, {after['secs']:.0f}s)."))

Chosen on validation: depth 4, learning rate 0.1 (733 trees, 238s).

## 4. Before and after on the test set (once each)

In [6]:
res = pd.DataFrame([test_row("before: J1 settings", before), test_row("after: chosen settings", after)]).set_index("model")
res["trees"] = [before["trees"], after["trees"]]
committed = pd.read_csv(ROOT / "reports" / "ablation.csv")
committed_raw = committed[(committed["model"] == "XGBoost") & (committed["features"] == "raw")].iloc[0]
display(res[["trees", "fit_seconds", "PR-AUC", "ROC-AUC", "precision", "recall", "F1", "FN", "FP"]]
        .style.format({"trees": "{:,}", "fit_seconds": "{:.0f}", "PR-AUC": "{:.4f}", "ROC-AUC": "{:.4f}", "precision": "{:.3f}",
                       "recall": "{:.3f}", "F1": "{:.3f}", "FN": "{:,}", "FP": "{:,}"}))
print(f"committed J1 ablation row (XGBoost, raw): PR-AUC {committed_raw['PR-AUC']:.4f}, FN {int(committed_raw['FN'])}, FP {int(committed_raw['FP'])}")
assert abs(res.loc["before: J1 settings", "PR-AUC"] - committed_raw["PR-AUC"]) < 5e-4, "the symptom no longer reproduces J1"

out = res.reset_index().rename(columns={"model": "run"})
out.to_csv(ROOT / "reports" / "m2_2" / "raw_convergence.csv", index=False)

,trees,fit_seconds,PR-AUC,ROC-AUC,precision,recall,F1,FN,FP
model,,,,,,,,,
before: J1 settings,"1,886",536,0.9289,0.9981,0.964,0.742,0.839,318,34
after: chosen settings,733,238,0.9295,0.9979,0.967,0.743,0.840,317,31


committed J1 ablation row (XGBoost, raw): PR-AUC 0.9289, FN 318, FP 34


## 5. Reading the result

- **The symptom was real.** With the J1 settings the raw-feature model needed 1,886 trees, and its validation curve was
  still creeping up (+0.0011 over its last 300 rounds), so it was converging slowly rather than having settled.
- **The fix.** Depth 4 with a learning rate of 0.1 instead of 0.03 reaches the same place in 733 trees, 61% fewer, and
  in 238 s instead of 536 s (56% less). Absolute seconds depend on the machine and its load: J1 recorded 258 s for the
  "before" fit, this run measured 536 s, so only the within-run comparison means anything.
- **Accuracy did not change.** Test PR-AUC 0.9295 against 0.9289 (+0.0006, well inside what a different random seed
  moves), 317 missed frauds against 318, 31 false alarms against 34. The "before" row reproduces the committed J1
  ablation number (0.9289) exactly, which the notebook asserts.
- **What this tells us about the J1 ablation.** Raw-feature XGBoost scores about 0.93 whether it is trained slowly or
  quickly, so the large gap to the engineered-feature model (0.997) is not an artefact of under-training. It reflects
  the balance-bookkeeping features doing the work, as J1 concluded.
- **What is not fixed.** The raw-feature model is still at 0.93, and 733 trees is not small. The "converged" test (stop
  at or below half the ceiling) was chosen before the runs and is a convenience, not a theory. J1's `ablation.csv` is
  left as committed; the new numbers are in `reports/m2_2/raw_convergence.csv`.